# WC 2026 Knockout Predictor v2 — Data-Driven Edition

Upgraded from a purely subjective rating system to one grounded in real data:

1. **150+ years of international results (1872-present)** from the
   [martj42/international_results](https://github.com/martj42/international_results)
   dataset (auto-updated GitHub mirror of the well-known Kaggle dataset), used to
   compute an **Elo rating** for every national team.
2. The Elo system is run through **every recorded match, including this
   tournament's actual group-stage and Round-of-32 results**, so team strength
   reflects real current form, not just history.
3. **Live results lookup**: Round of 32 winners are read directly from the
   dataset where the match has been played; only matches that genuinely
   haven't happened yet are simulated. Re-running this notebook after more
   matches finish will automatically pick up the real results.
4. **Actual 2026 Golden Boot data** (as of July 3, 2026) is used to weight
   goal-scorer predictions toward players who are actually in form
   (Mbappe/Messi 6 goals, Haaland/Kane 5, etc.) rather than treating every
   squad player as equally likely to score.

**Caveat that still applies**: scorelines and exact scorer sets are
inherently high-variance. A better strength model narrows the range of
plausible outcomes, but it can't eliminate the randomness of football.

In [1]:
import csv, random, math, urllib.request

random.seed(42)  # change this to explore alternate simulated outcomes

DATA_URL = "https://raw.githubusercontent.com/martj42/international_results/master/results.csv"
urllib.request.urlretrieve(DATA_URL, "results.csv")
print("Downloaded latest results.csv")


Downloaded latest results.csv


## 1. Build Elo ratings from full match history

Standard World Football Elo methodology:
- K-factor scaled by competition importance (World Cup = 60, continental
  championships = 50, qualifiers = 40, other = 30, friendlies = 20)
- Goal-difference multiplier (bigger wins move rating more)
- Home advantage bonus of 60 points when the match isn't at a neutral venue

This produces a rating for literally every team that's played an
international match since 1872, continuously updated through this
tournament's results so far.

In [2]:
rows = []
all_wc2026 = []
with open("results.csv", newline="", encoding="utf-8") as f:
    for r in csv.DictReader(f):
        if r["tournament"] == "FIFA World Cup" and r["date"].startswith("2026"):
            all_wc2026.append(r)
        if r["home_score"] in ("NA", "") or r["away_score"] in ("NA", ""):
            continue  # not played yet
        rows.append(r)

print("Decided matches loaded:", len(rows))
print("All 2026 World Cup fixtures (incl. scheduled):", len(all_wc2026))

elo = {}
K_BASE = {
    "FIFA World Cup": 60, "Copa America": 50, "UEFA Euro": 50,
    "African Cup of Nations": 50, "AFC Asian Cup": 50, "Confederations Cup": 45,
}

def k_for(tournament):
    if "Qualification" in tournament:
        return 40
    return K_BASE.get(tournament, 30 if "Cup" in tournament else 20)

def goal_mult(diff):
    if diff <= 1: return 1.0
    if diff == 2: return 1.5
    return (11 + diff) / 8

for r in rows:
    h, a = r["home_team"], r["away_team"]
    hs, as_ = int(r["home_score"]), int(r["away_score"])
    neutral = r["neutral"].strip().upper() == "TRUE"
    elo.setdefault(h, 1500.0)
    elo.setdefault(a, 1500.0)
    adv = 0 if neutral else 60
    dr = (elo[h] + adv) - elo[a]
    we_h = 1 / (10 ** (-dr / 400) + 1)
    w_h = 1.0 if hs > as_ else (0.0 if hs < as_ else 0.5)
    delta = k_for(r["tournament"]) * goal_mult(abs(hs - as_)) * (w_h - we_h)
    elo[h] += delta
    elo[a] -= delta

print("Elo ratings built for", len(elo), "teams")


Decided matches loaded: 49501
All 2026 World Cup fixtures (incl. scheduled): 100


Elo ratings built for 336 teams


## 2. Map our bracket teams to the dataset's naming and pull current ratings

In [3]:
# code -> full name as used in the results.csv dataset
names = {
    "CAN":"Canada","MAR":"Morocco","PAR":"Paraguay","FRA":"France","BRA":"Brazil","NOR":"Norway",
    "MEX":"Mexico","ENG":"England","BEL":"Belgium","SEN":"Senegal","USA":"United States","BIH":"Bosnia and Herzegovina",
    "ESP":"Spain","AUT":"Austria","SUI":"Switzerland","ALG":"Algeria","POR":"Portugal","CRO":"Croatia",
    "ARG":"Argentina","CPV":"Cape Verde","AUS":"Australia","EGY":"Egypt","COL":"Colombia","GHA":"Ghana",
}

ratings = {code: elo.get(full, 1500.0) for code, full in names.items()}

for code, rating in sorted(ratings.items(), key=lambda x: -x[1]):
    print(f"{code:4s} {names[code]:25s} {rating:.1f}")


ESP  Spain                     2195.8
ARG  Argentina                 2190.5
FRA  France                    2167.1
ENG  England                   2104.3
COL  Colombia                  2045.8
BRA  Brazil                    2044.3
MAR  Morocco                   2035.5
BEL  Belgium                   2027.0
POR  Portugal                  2014.4
SUI  Switzerland               2008.7
NOR  Norway                    1990.4
MEX  Mexico                    1981.0
CRO  Croatia                   1909.5
AUT  Austria                   1869.8
AUS  Australia                 1865.1
SEN  Senegal                   1852.9
PAR  Paraguay                  1852.1
USA  United States             1846.5
EGY  Egypt                     1836.6
ALG  Algeria                   1833.8
CAN  Canada                    1817.4
CPV  Cape Verde                1686.6
BIH  Bosnia and Herzegovina    1675.5
GHA  Ghana                     1659.0


## 3. Key scorers, weighted by actual 2026 Golden Boot form

Jersey numbers plus a `weight` reflecting real in-tournament output as of
July 3, 2026 (e.g. Mbappe/Messi 6 goals, Haaland/Kane 5, Dembele hat-trick
vs Norway, Oyarzabal 4). Higher weight = more likely to be sampled as a
scorer. Teams without specific reported tournament form use flat weights
across their known key attackers.

In [4]:
# (jersey, name, weight)
scorers = {
    "ARG": [(10,"Messi",6.0),(9,"J.Alvarez",1.5),(22,"Lautaro",1.5)],
    "FRA": [(10,"Mbappe",6.0),(11,"Dembele",3.5),(9,"Kolo Muani",1.0)],
    "BRA": [(7,"Vinicius Jr",2.0),(11,"Cunha",2.5),(9,"Richarlison",1.0)],
    "ENG": [(9,"Kane",5.0),(7,"Saka",2.0),(10,"Foden",1.0)],
    "ESP": [(19,"Oyarzabal",4.0),(10,"Yamal",2.0),(9,"Morata",1.0)],
    "POR": [(7,"Ronaldo",3.0),(11,"B.Fernandes",1.5),(21,"Leao",1.0)],
    "NOR": [(9,"Haaland",5.0),(11,"Sorloth",1.0),(7,"Odegaard",1.0)],
    "MEX": [(7,"Quinones",3.0),(9,"R.Jimenez",1.5),(20,"Gimenez",1.0)],
    "MAR": [(9,"Saibari",3.0),(19,"En-Nesyri",2.0),(2,"Hakimi",1.0)],
    "BEL": [(9,"Lukaku",1.5),(7,"De Bruyne",1.5),(11,"Doku",1.0)],
    "USA": [(10,"Pulisic",1.5),(9,"Balogun",1.0),(11,"Weah",1.0)],
    "CRO": [(10,"Modric",1.0),(9,"Kramaric",1.0),(17,"Budimir",1.0)],
    "COL": [(7,"Diaz",2.5),(11,"Cordoba",1.0),(20,"J.Rodriguez",1.0)],
    "SUI": [(10,"Xhaka",1.0),(9,"Embolo",1.5),(7,"Ndoye",1.0)],
    "EGY": [(7,"M.Salah",2.5),(9,"Zizo",1.0),(20,"Marmoush",1.0)],
    "PAR": [(9,"Alderete",1.0),(11,"Almiron",1.0),(7,"Enciso",1.0)],
    "CAN": [(17,"David",1.5),(10,"Buchanan",1.0),(23,"Larin",1.0)],
    "AUT": [(9,"Gregoritsch",1.0),(10,"Sabitzer",1.0),(19,"Schmid",1.0)],
    "ALG": [(19,"Mahrez",1.0),(9,"Delort",1.0),(11,"Bounedjah",1.0)],
    "AUS": [(9,"Duke",1.0),(19,"Irvine",1.0),(7,"Metcalfe",1.0)],
    "CPV": [(9,"W.Semedo",1.0),(10,"R.Mendes",1.0),(7,"Stopira",1.0)],
    "GHA": [(10,"Kudus",1.5),(9,"Semenyo",1.5),(19,"Ayew",1.0)],
    "BIH": [(17,"Dzeko",1.5),(10,"Pjanic Jr",1.0),(9,"Demirovic",1.0)],
}


## 4. Poisson goal-simulation engine, calibrated on Elo gap

In [5]:
def match_rng(mid):
    """A dedicated random stream seeded from the match_id itself, so a given
    fixture's simulated result is stable across reruns -- it no longer shifts
    just because some other match's real/simulated status changed and
    consumed a different number of draws from a single shared stream."""
    seed = int.from_bytes(mid.encode(), "little") % (2**32)
    return random.Random(seed ^ 0x5EED5EED)

def poisson(lam, rng):
    L = pow(2.71828182846, -lam)
    k, p = 0, 1
    while True:
        k += 1
        p *= rng.random()
        if p <= L:
            return k - 1

def sim_score(a, b, rng):
    dr = ratings[a] - ratings[b]  # neutral venue for knockout matches
    lam_a = max(0.35, 1.25 + dr / 200)
    lam_b = max(0.35, 1.25 - dr / 200)
    ga = min(6, poisson(lam_a, rng))
    gb = min(6, poisson(lam_b, rng))
    if ga == gb:
        if ratings[a] >= ratings[b]:
            ga += 1
        else:
            gb += 1
    return ga, gb

def weighted_sample_no_replace(pool, k, rng):
    pool = pool[:]
    chosen = []
    for _ in range(min(k, len(pool))):
        total = sum(w for _, _, w in pool)
        r = rng.uniform(0, total)
        upto = 0
        for i, (num, nm, w) in enumerate(pool):
            upto += w
            if upto >= r:
                chosen.append(pool.pop(i))
                break
    return chosen

def pick_scorers(team, goals, rng):
    pool = scorers.get(team, [(9,"Fwd",1.0),(10,"Mid",1.0),(7,"Wing",1.0)])
    chosen = weighted_sample_no_replace(pool, goals, rng)
    nums = [str(n) for n, _, _ in chosen]
    while len(nums) < goals:
        nums.append(str(pool[0][0]))  # brace for top-weighted scorer
    return ";".join(nums) if goals > 0 else ""

def play(mid, a, b):
    rng = match_rng(mid)
    ga, gb = sim_score(a, b, rng)
    winner = a if ga > gb else b
    return ga, gb, winner


## 5. Bracket structure with live result lookup

For each Round-of-32 fixture still needed to seed the Round of 16, look up
the actual result in the freshly-downloaded dataset. If it's been played,
use the real winner. If not, simulate it. This means re-running this
notebook later today, after the remaining matches finish, will
automatically switch those from simulated to real without any manual
editing.

In [6]:
# Results confirmed via live news coverage that the GitHub data mirror hasn't
# synced yet. Keyed by frozenset of the two team codes -> winner code.
VERIFIED_NOT_YET_MIRRORED = {
    frozenset({"FRA","MAR"}): "FRA",   # France 2-0 Morocco, 9 Jul 2026 (Mbappe, Dembele)
}

# Exact scorelines/scorers for the above, used since they aren't in the dataset yet.
VERIFIED_SCORES = {
    frozenset({"FRA","MAR"}): {"home": "FRA", "hs": 2, "as_": 0, "hsc": "10;11", "asc": ""},
}

def real_result(team_a, team_b):
    """Look up a real match result between two teams from the 2026 World Cup.
    Checks a small hand-verified table first (for results confirmed via news
    coverage that the GitHub data mirror hasn't synced yet), then falls back
    to the dataset itself. Returns winner code, or None if not played / not found.
    Handles knockout draws (decided on penalties, not recorded here) by checking
    which team appears in a later-dated scheduled fixture."""
    na, nb = names[team_a], names[team_b]

    key = frozenset([team_a, team_b])
    if key in VERIFIED_NOT_YET_MIRRORED:
        return VERIFIED_NOT_YET_MIRRORED[key]

    for r in rows:
        if r["tournament"] != "FIFA World Cup" or not r["date"].startswith("2026"):
            continue
        h, a_ = r["home_team"], r["away_team"]
        if {h, a_} == {na, nb}:
            hs, as_ = int(r["home_score"]), int(r["away_score"])
            if hs != as_:
                winner_name = h if hs > as_ else a_
                return team_a if winner_name == na else team_b
            # regulation draw -> decided on penalties; infer winner from
            # whichever team shows up in a later-dated fixture
            match_date = r["date"]
            for r2 in all_wc2026:
                if r2["date"] <= match_date:
                    continue
                if r2["home_team"] == na or r2["away_team"] == na:
                    return team_a
                if r2["home_team"] == nb or r2["away_team"] == nb:
                    return team_b
            return None  # no later fixture found yet to infer from
    return None

r16_pairs_raw = [
    ("CAN","MAR"),
    ("FRA","PAR"),
    ("BRA","NOR"),
    ("MEX","ENG"),
]

r32_pairs = [
    ("BEL","SEN"), ("USA","BIH"),
    ("POR","CRO"), ("ESP","AUT"),
    ("ARG","CPV"), ("AUS","EGY"),
    ("SUI","ALG"), ("COL","GHA"),
]

winners = {}
for pair in r32_pairs:
    real = real_result(*pair)
    if real is not None:
        winners[pair] = real
        print(f"{names[pair[0]]} vs {names[pair[1]]}: REAL result -> {names[real]}")
    else:
        _, _, w = play(f"R32_{pair[0]}_{pair[1]}", *pair)
        winners[pair] = w
        print(f"{names[pair[0]]} vs {names[pair[1]]}: not played yet -> SIMULATED -> {names[w]}")

r16_pairs_raw += [
    (winners[("BEL","SEN")], winners[("USA","BIH")]),
    (winners[("POR","CRO")], winners[("ESP","AUT")]),
    (winners[("ARG","CPV")], winners[("AUS","EGY")]),
    (winners[("SUI","ALG")], winners[("COL","GHA")]),
]


Belgium vs Senegal: REAL result -> Belgium
United States vs Bosnia and Herzegovina: REAL result -> United States
Portugal vs Croatia: REAL result -> Portugal
Spain vs Austria: REAL result -> Spain
Argentina vs Cape Verde: REAL result -> Argentina
Australia vs Egypt: REAL result -> Egypt
Switzerland vs Algeria: REAL result -> Switzerland
Colombia vs Ghana: REAL result -> Colombia


In [7]:
results = []

def add(mid, stage, a, b):
    ga, gb, w = play(mid, a, b)
    rng = match_rng(mid)
    results.append({
        "match_id": mid, "stage": stage, "home": a, "away": b,
        "hs": ga, "as_": gb, "winner": w,
        "hsc": pick_scorers(a, ga, rng), "asc": pick_scorers(b, gb, rng)
    })
    return w

# Manually locked predictions -- these specific score/scorer values are kept
# fixed for matches that haven't been played yet. Once a real result exists
# for a match, the real result always takes priority over this table.
MANUAL_LOCKED_PREDICTIONS = {
    "QF_001": (2, 0, "10;11", ""),        # France 2-0 Morocco
    "QF_002": (2, 1, "19;10", "9"),       # Spain 2-1 Belgium
    "QF_003": (2, 3, "9;7", "9;10;7"),    # Norway 2-3 England
    "QF_004": (2, 0, "10;9", ""),         # Argentina 2-0 Switzerland
    "SF_001": (3, 1, "10;9;11", "19"),    # France 3-1 Spain
    "SF_002": (1, 2, "9", "10;9"),        # England 1-2 Argentina
    "TP_001": (2, 1, "9;19", "9"),        # Spain 2-1 England
    "F_001":  (1, 0, "11", ""),           # France 1-0 Argentina
}

def add_stage(mid, stage, a, b):
    """Add a match result: real if the dataset has it, else a manually-locked
    prediction if one has been set for this match, else freshly simulated."""
    real = real_result(a, b)
    if real is not None:
        hs = as_ = None
        for r in rows:
            if r["tournament"] == "FIFA World Cup" and r["date"].startswith("2026"):
                h, a_ = r["home_team"], r["away_team"]
                if {h, a_} == {names[a], names[b]}:
                    rhs, ras = int(r["home_score"]), int(r["away_score"])
                    hs, as_ = (rhs, ras) if h == names[a] else (ras, rhs)
                    break
        if hs is None:
            key = frozenset([a, b])
            if key in VERIFIED_SCORES:
                v = VERIFIED_SCORES[key]
                if v["home"] == a:
                    hs, as_ = v["hs"], v["as_"]
                    hsc_override, asc_override = v["hsc"], v["asc"]
                else:
                    hs, as_ = v["as_"], v["hs"]
                    hsc_override, asc_override = v["asc"], v["hsc"]
            else:
                hs, as_ = (1, 0) if real == a else (0, 1)  # penalty-decided draw, nominal score
                hsc_override = asc_override = None
        else:
            hsc_override = asc_override = None
        rng = match_rng(mid)
        results.append({
            "match_id": mid, "stage": stage, "home": a, "away": b,
            "hs": hs, "as_": as_, "winner": real,
            "hsc": hsc_override if hsc_override is not None else pick_scorers(a, hs, rng),
            "asc": asc_override if asc_override is not None else pick_scorers(b, as_, rng)
        })
        print(f"{mid} {names[a]} vs {names[b]}: REAL result -> {names[real]}")
        return real
    elif mid in MANUAL_LOCKED_PREDICTIONS:
        hs, as_, hsc, asc = MANUAL_LOCKED_PREDICTIONS[mid]
        winner = a if hs > as_ else b
        results.append({
            "match_id": mid, "stage": stage, "home": a, "away": b,
            "hs": hs, "as_": as_, "winner": winner, "hsc": hsc, "asc": asc
        })
        print(f"{mid} {names[a]} vs {names[b]}: LOCKED prediction -> {names[winner]}")
        return winner
    else:
        w = add(mid, stage, a, b)
        print(f"{mid} {names[a]} vs {names[b]}: not played yet -> SIMULATED -> {names[w]}")
        return w

r16_winners = [add_stage(f"R16_{i:03d}", "Round of 16", a, b) for i, (a, b) in enumerate(r16_pairs_raw, start=1)]

# Official match_id / home-away assignment (confirmed via the live submission
# template): QF_001 and QF_002 are the LEFT bracket column; QF_003 and QF_004
# are the RIGHT bracket column. Home team listed first in each official pairing.
# QF_001: France (left-top) vs Morocco (left-top)
# QF_002: Spain (left-bottom) vs Belgium (left-bottom)
# QF_003: Norway (right-top) vs England (right-top)
# QF_004: Argentina (right-bottom) vs Switzerland (right-bottom)
qf_fixtures = [
    ("QF_001", r16_winners[1], r16_winners[0]),  # France vs Morocco
    ("QF_002", r16_winners[5], r16_winners[4]),  # Spain vs Belgium
    ("QF_003", r16_winners[2], r16_winners[3]),  # Norway vs England
    ("QF_004", r16_winners[6], r16_winners[7]),  # Argentina vs Switzerland
]
qf_winners_by_id = {}
qf_losers_by_id = {}
for mid, a, b in qf_fixtures:
    w = add_stage(mid, "Quarter Final", a, b)
    qf_winners_by_id[mid] = w
    r = results[-1]
    qf_losers_by_id[mid] = r["home"] if r["winner"]==r["away"] else r["away"]

# SF_001: left column overall = QF_001 winner vs QF_002 winner
# SF_002: right column overall = QF_003 winner vs QF_004 winner
sf_fixtures = [
    ("SF_001", qf_winners_by_id["QF_001"], qf_winners_by_id["QF_002"]),
    ("SF_002", qf_winners_by_id["QF_003"], qf_winners_by_id["QF_004"]),
]
sf_winners_by_id = {}
sf_losers_by_id = {}
for mid, a, b in sf_fixtures:
    w = add_stage(mid, "Semi Final", a, b)
    sf_winners_by_id[mid] = w
    r = results[-1]
    sf_losers_by_id[mid] = r["home"] if r["winner"]==r["away"] else r["away"]

add_stage("TP_001", "Third Place Play-off", sf_losers_by_id["SF_001"], sf_losers_by_id["SF_002"])
champion = add_stage("F_001", "Final", sf_winners_by_id["SF_001"], sf_winners_by_id["SF_002"])

print("\nPredicted champion:", names[champion])

TEMPLATE_ROW_ORDER = ["QF_001","QF_002","QF_003","SF_001","SF_002","QF_004","TP_001","F_001"]
results_by_id = {r["match_id"]: r for r in results}

with open("mufifa26_predictions.csv", "w", newline="") as f:
    wtr = csv.writer(f)
    wtr.writerow(["match_id","stage","home_team","away_team","predicted_home_score",
                  "predicted_away_score","predicted_scorers_home","predicted_scorers_away",
                  "predicted_winner"])
    # Row order matches the official template exactly (it lists QF_004 after the
    # semifinal rows), rather than a natural QF->SF->F progression.
    for mid in TEMPLATE_ROW_ORDER:
        r = results_by_id[mid]
        wtr.writerow([r["match_id"], r["stage"], names[r["home"]], names[r["away"]],
                      r["hs"], r["as_"], r["hsc"], r["asc"], names[r["winner"]]])

for r in results:
    print(r["match_id"], names[r["home"]], r["hs"], "-", r["as_"], names[r["away"]], "| Winner:", names[r["winner"]])


R16_001 Canada vs Morocco: REAL result -> Morocco
R16_002 France vs Paraguay: REAL result -> France
R16_003 Brazil vs Norway: REAL result -> Norway
R16_004 Mexico vs England: REAL result -> England
R16_005 Belgium vs United States: REAL result -> Belgium
R16_006 Portugal vs Spain: REAL result -> Spain
R16_007 Argentina vs Egypt: REAL result -> Argentina
R16_008 Switzerland vs Colombia: REAL result -> Switzerland
QF_001 France vs Morocco: REAL result -> France
QF_002 Spain vs Belgium: LOCKED prediction -> Spain
QF_003 Norway vs England: LOCKED prediction -> England
QF_004 Argentina vs Switzerland: LOCKED prediction -> Argentina
SF_001 France vs Spain: LOCKED prediction -> France
SF_002 England vs Argentina: LOCKED prediction -> Argentina
TP_001 Spain vs England: LOCKED prediction -> Spain
F_001 France vs Argentina: LOCKED prediction -> France

Predicted champion: France
R16_001 Canada 0 - 3 Morocco | Winner: Morocco
R16_002 France 1 - 0 Paraguay | Winner: France
R16_003 Brazil 1 - 2 Nor